In [1]:
import subprocess
from pathlib import Path

import pandas as pd
from Bio import SeqIO
from tqdm import tqdm

from utils.kaptive_host_klocus_extractor import HostKLocusExtractor
from utils.phage_pipeline_runner import run_phage_phanotate, run_phage_rbpdetect
from utils.recover_bacformer_positions import recover_host_positions, recover_phage_positions
from utils.columnwise_mean_aggregation import aggregate_columnwise_mean
from utils.compute_cosine_similarities import compute_phage_host_similarities

/home/biffon/.conda/envs/phage_host_env_gnn_model/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# MODEL CONFIGS 
MODEL_CONFIGS = {
    "ESM2": {
        "conda_env": "esm2_env",
        "script": "utils/generate_esm2_embeddings.py",
        "tag": "esm2",
        "extra_args": ["--no-capture-output"],
    },
    "ESMC": {
        "conda_env": "esm3_env",
        "script": "utils/generate_esmc_embeddings.py",
        "tag": "esmc",
        "extra_args": ["--no-capture-output"],
    },
    "LucaOne": {
        "conda_env": "new_lucaone_app_env",
        "script": "utils/generate_lucaone_embeddings.py",
        "tag": "lucaone",
        "extra_args": ["--no-capture-output"],
    },
    "BacFormer": {
        "conda_env": "bacformer_env",
        "script": "utils/generate_bacformer_embeddings.py",
        "tag": "bacformer",
        "extra_args": ["--no-capture-output"],
    },
    "AF3": {
        "conda_env": None,
        "script": "utils/structural_features_extraction_esm_IF1.py",
        "tag": "af3",
        "extra_args": ["--no-capture-output"],
    },
}

KAPTIVE_DB = "kpsc_k"
RBP_MIN_LEN = 200
RBP_MAX_LEN = 1500
MAKE_COUNT_PLOTS = True

In [3]:
# USER INPUTS
BASE_DIR = Path("datasets/121strains_74phages")
INPUT_MODE = "genomes"  # set to "genomes" or "proteins" to start from raw genomes or existing protein FASTAs (K-locus or RBP)
MODEL_NAMES = ["ESM2", "ESMC", "LucaOne", "BacFormer", "AF3"] # Select representation models to run

# RAW GENOME DIRS
HOST_GENOME_DIR = BASE_DIR / "genomes/hosts/host_kp_genomes"
PHAGE_GENOME_DIR = BASE_DIR / "genomes/phages/phage_genomes"

# AF3-predicted protein structures (supplied by the user)
AF3_HOST_CIF_DIR = BASE_DIR / "protein_structures/Locibase_cifs"
AF3_PHAGE_CIF_DIR = BASE_DIR / "protein_structures/RBPbase_cifs"

# Derived processing and output paths
KAPTIVE_OUTDIR = BASE_DIR / "processing/hosts/kaptive_out"
PHANOTATE_OUTDIR = BASE_DIR / "processing/phages/phanotate_out"
RBP_OUTDIR = BASE_DIR / "processing/phages/rbp_out"

BACFORMER_HOST_DIR = BASE_DIR / "processing/hosts/bacformer"
BACFORMER_PHAGE_DIR = BASE_DIR / "processing/phages/bacformer"

EMBEDDING_DIR = BASE_DIR / "embeddings"
SIMILARITY_DIR = BASE_DIR / "cosine_similarity"

HOST_PROTEIN_FASTA = KAPTIVE_OUTDIR / "all_strain_kl_proteins.faa"
PHAGE_PROTEIN_FASTA = RBP_OUTDIR / "all_phage_rbps.faa"
KAPTIVE_PROTEIN_INDEX = KAPTIVE_OUTDIR / "strain_protein_index.tsv"

In [4]:
# Preparing K-locus and RBP protein FASTAs
def prepare_protein_fastas():
    """Generate protein FASTAs from genomes or use existing protein FASTAs."""
    if INPUT_MODE == "genomes":
        print("\nProcessing host genomes...")
        HostKLocusExtractor(
            genome_dir=HOST_GENOME_DIR,
            outdir=KAPTIVE_OUTDIR,
            kaptive_db=KAPTIVE_DB,
            show_progress=True,
        ).run()

        print("\nProcessing phage genomes...")
        run_phage_phanotate(PHAGE_GENOME_DIR, PHANOTATE_OUTDIR)
        run_phage_rbpdetect(PHANOTATE_OUTDIR, RBP_OUTDIR, min_len=RBP_MIN_LEN, max_len=RBP_MAX_LEN)

    elif INPUT_MODE != "proteins":
        raise ValueError("INPUT_MODE must be 'genomes' or 'proteins'.")

    for fasta, name in [(HOST_PROTEIN_FASTA, "Host K-locus"), (PHAGE_PROTEIN_FASTA, "Phage RBP")]:
        if not fasta.exists():
            raise FileNotFoundError(f"{name} FASTA not found: {fasta}")

        n_proteins = sum(1 for _ in SeqIO.parse(str(fasta), "fasta"))
        if n_proteins == 0:
            raise ValueError(f"No proteins found in: {fasta}")
        print(f"{name} proteins: {n_proteins}")

    return HOST_PROTEIN_FASTA, PHAGE_PROTEIN_FASTA

# Preparing Bacformer genomic positions
def prepare_bacformer_positions(host_fasta, phage_fasta):
    """Recover genomic positions for host K-locus proteins and phage RBPs."""

    if not KAPTIVE_PROTEIN_INDEX.exists():
        raise FileNotFoundError(f"Kaptive protein index not found: {KAPTIVE_PROTEIN_INDEX}")

    print("\nRecovering BacFormer host protein positions...")
    host_positions = recover_host_positions(
        genome_dir=HOST_GENOME_DIR,
        protein_fasta=host_fasta,
        protein_index_tsv=KAPTIVE_PROTEIN_INDEX,
        out_dir=BACFORMER_HOST_DIR,
    )

    print("\nRecovering BacFormer phage RBP positions...")
    phage_positions = recover_phage_positions(
        genome_dir=PHAGE_GENOME_DIR,
        rbp_fasta=phage_fasta,
        out_dir=BACFORMER_PHAGE_DIR,
    )
    return Path(host_positions), Path(phage_positions)


# Validating AF3 CIF structures
def validate_af3_structures(host_fasta, phage_fasta):
    """Check that AF3 CIF structures exist for all required proteins."""
    if not AF3_HOST_CIF_DIR.exists():
        raise FileNotFoundError(f"AF3 host CIF directory not found: {AF3_HOST_CIF_DIR}")

    if not AF3_PHAGE_CIF_DIR.exists():
        raise FileNotFoundError(f"AF3 phage CIF directory not found: {AF3_PHAGE_CIF_DIR}")

    host_ids = {rec.id for rec in SeqIO.parse(str(host_fasta), "fasta")}
    phage_ids = {rec.id for rec in SeqIO.parse(str(phage_fasta), "fasta")}

    host_cif_ids = {p.stem for p in AF3_HOST_CIF_DIR.glob("*.cif")}
    phage_cif_ids = {p.stem for p in AF3_PHAGE_CIF_DIR.glob("*.cif")}

    missing_hosts = sorted(host_ids - host_cif_ids)
    missing_phages = sorted(phage_ids - phage_cif_ids)

    print(f"\nAF3 host structures : {len(host_cif_ids)} / {len(host_ids)}")
    print(f"AF3 phage structures: {len(phage_cif_ids)} / {len(phage_ids)}")

    if missing_hosts:
        raise ValueError(
            f"Missing AF3 structures for {len(missing_hosts)} host proteins. "
            f"Examples: {missing_hosts[:10]}"
        )

    if missing_phages:
        raise ValueError(
            f"Missing AF3 structures for {len(missing_phages)} phage RBPs. "
            f"Examples: {missing_phages[:10]}"
        )

    return AF3_HOST_CIF_DIR, AF3_PHAGE_CIF_DIR

In [5]:
#generating protein-level embeddings using the selected model
def generate_embeddings(model_name, input_file, outdir):
    cfg = MODEL_CONFIGS[model_name]
    input_file = Path(input_file)
    outdir.mkdir(parents=True, exist_ok=True)
    if cfg["conda_env"] is None:
        cmd = [
            "python",
            cfg["script"],
            str(input_file),
            str(outdir),
        ]
    else:
        cmd = [
            "conda",
            "run",
            *cfg["extra_args"],
            "-n",
            cfg["conda_env"],
            "python",
            cfg["script"],
            str(input_file),
            str(outdir),
        ]

    print(f"\nGenerating {model_name} embeddings")
    print(f"Input : {input_file}")
    print(f"Output: {outdir}")

    if model_name not in {"BacFormer", "AF3"}:
        n_proteins = sum(1 for _ in SeqIO.parse(str(input_file), "fasta"))
        print(f"Input proteins: {n_proteins}")

    subprocess.run(cmd, check=True)

    if model_name == "BacFormer":
        out_name = input_file.stem.replace("_positions", "_embeddings") + ".csv"
        out_csv = outdir / out_name

    elif model_name == "AF3":
        out_csv = outdir / f"{input_file.name}_embeddings.csv"

    else:
        out_csv = outdir / f"{input_file.stem}.csv"

    if not out_csv.exists():
        raise FileNotFoundError(f"Embedding file not found: {out_csv}")

    df = pd.read_csv(out_csv)
    if df.empty:
        raise ValueError(f"Empty embedding file: {out_csv}")

    print(f"{model_name} completed: {df.shape[0]} proteins × {df.shape[1] - 1} dimensions")
    return out_csv


# aggregating protein embeddings (mean-pool embeddings into one vector per host or phage)
def aggregate_embeddings(model_name, host_csv, phage_csv):
    tag = MODEL_CONFIGS[model_name]["tag"]
    outdir = EMBEDDING_DIR / model_name / "final_embeddings_colmean"
    print(f"\nAggregating {model_name} embeddings...")

    host_out = aggregate_columnwise_mean(
        emb_path=host_csv,
        outdir=outdir,
        entity_type="host",
        output_prefix=f"host_klocus_{tag}",
        make_plot=MAKE_COUNT_PLOTS,
    )

    phage_out = aggregate_columnwise_mean(
        emb_path=phage_csv,
        outdir=outdir,
        entity_type="phage",
        output_prefix=f"phage_rbp_{tag}",
        make_plot=MAKE_COUNT_PLOTS,
    )

    return (
        Path(host_out["embeddings_csv"]),
        Path(phage_out["embeddings_csv"]),
    )


# computing within-type cosine similarities (host-host and phage-phage cosine similarities)
def compute_similarities(model_name, host_csv, phage_csv):

    host_df = pd.read_csv(host_csv, index_col=0)
    phage_df = pd.read_csv(phage_csv, index_col=0)

    phage_sim, host_sim, phage_sim_csv, host_sim_csv = (
        compute_phage_host_similarities(
            phage_emb_df=phage_df,
            host_emb_df=host_df,
            out_dir=SIMILARITY_DIR / model_name,
            model_name=model_name,
            dataset_tag=BASE_DIR.name,
            suffix="canonical",
            verbose=False,
        )
    )

    print(f"{model_name} host embeddings : {host_df.shape}")
    print(f"{model_name} phage embeddings: {phage_df.shape}")
    print(f"{model_name} host similarity : {host_sim.shape}")
    print(f"{model_name} phage similarity: {phage_sim.shape}")
    return Path(host_sim_csv), Path(phage_sim_csv)

In [6]:
# processing one representation model (generate and aggregate embeddings, and compute similarities)
def process_model(model_name, host_fasta, phage_fasta, bacformer_positions=None, af3_inputs=None):

    print("\n" + "=" * 20)
    print(f"PROCESSING {model_name}")

    model_dir = EMBEDDING_DIR / model_name
    if model_name == "BacFormer":
        if bacformer_positions is None:
            raise ValueError("BacFormer position files were not provided.")
        host_input, phage_input = bacformer_positions

    elif model_name == "AF3":
        if af3_inputs is None:
            raise ValueError("AF3 CIF directories were not provided.")
        host_input, phage_input = af3_inputs

    else:
        host_input, phage_input = host_fasta, phage_fasta

    host_protein_csv = generate_embeddings(model_name, host_input, model_dir / "host_klocus_protein_embeddings")
    phage_protein_csv = generate_embeddings(model_name, phage_input, model_dir / "phage_rbp_protein_embeddings")
    host_csv, phage_csv = aggregate_embeddings(model_name, host_protein_csv, phage_protein_csv)
    host_sim_csv, phage_sim_csv = compute_similarities(model_name, host_csv, phage_csv)

    return {
        "host_embeddings": host_csv,
        "phage_embeddings": phage_csv,
        "host_similarity": host_sim_csv,
        "phage_similarity": phage_sim_csv,
    }

In [ ]:
# Main data processing pipeline

print("=" * 20)
print("PHAGE-HOST DATA PROCESSING")
print("Dataset:", BASE_DIR)
print(f"Input mode: {INPUT_MODE}; Models: {MODEL_NAMES}")

host_fasta, phage_fasta = prepare_protein_fastas()

bacformer_positions = None
af3_inputs = None

if "BacFormer" in MODEL_NAMES:
    bacformer_positions = prepare_bacformer_positions(host_fasta, phage_fasta)

if "AF3" in MODEL_NAMES:
    af3_inputs = validate_af3_structures(host_fasta, phage_fasta)

outputs = {}
for model_name in tqdm(MODEL_NAMES, desc="Representation models", unit="model"):
    outputs[model_name] = process_model(
        model_name,
        host_fasta,
        phage_fasta,
        bacformer_positions=bacformer_positions,
        af3_inputs=af3_inputs,
    )

print("\n" + "=" * 70)
print("PROCESSING COMPLETED")

for model_name, files in outputs.items():
    print(f"\n{model_name}")
    print("Host embeddings :", files["host_embeddings"])
    print("Phage embeddings:", files["phage_embeddings"])
    print("Host similarity :", files["host_similarity"])
    print("Phage similarity:", files["phage_similarity"])

print("\nReady for traininvg, inference or local adaptation.")

PHAGE-HOST DATA PROCESSING
Dataset: datasets/121strains_74phages
Input mode: genomes; Models: ['ESM2', 'ESMC', 'LucaOne', 'BacFormer', 'AF3']

Processing host genomes...
1/6: Collecting host genomes
2/6: Preparing Kaptive inputs


Preparing Kaptive inputs:  31%|███       | 37/121 [00:00<00:00, 362.73it/s]

Preparing Kaptive inputs: 100%|██████████| 121/121 [00:00<00:00, 364.34it/s]


Host genomes found: 121
3/6: Running Kaptive

Running: kaptive assembly kpsc_k datasets/121strains_74phages/processing/hosts/kaptive_out/kaptive_input_genomes/19101412.fna ...
Log: datasets/121strains_74phages/processing/hosts/kaptive_out/kaptive_assembly.log

Running: kaptive convert kpsc_k datasets/121strains_74phages/processing/hosts/kaptive_out/kaptive_results.json ...
Log: datasets/121strains_74phages/processing/hosts/kaptive_out/kaptive_convert.log
4/6: Reading Kaptive results
Kaptive rows before filtering: 121
Mapped KL calls retained: 121
5/6: Writing final FASTA files


Writing final K-locus FASTAs: 100%|██████████| 121/121 [00:00<00:00, 9022.25it/s]


...

AF3 completed: 2870 proteins × 512 dimensions

Generating AF3 embeddings
Input : datasets/121strains_74phages/protein_structures/RBPbase_cifs
Output: datasets/121strains_74phages/embeddings/AF3/phage_rbp_protein_embeddings


ESM-IF1 embedding: 100%|██████████| 163/163 [00:14<00:00, 10.96protein/s]


AF3 completed: 163 proteins × 512 dimensions

Aggregating AF3 embeddings...
Host aggregation: 121 entities × 512 dimensions


Representation models: 100%|██████████| 5/5 [1:10:35<00:00, 847.08s/model] 

Phage aggregation: 74 entities × 512 dimensions
AF3 host embeddings : (121, 512)
AF3 phage embeddings: (74, 512)
AF3 host similarity : (121, 121)
AF3 phage similarity: (74, 74)

PROCESSING COMPLETED

ESM2
Host embeddings : datasets/121strains_74phages/embeddings/ESM2/final_embeddings_colmean/host_klocus_esm2_colmean_embeddings.csv
Phage embeddings: datasets/121strains_74phages/embeddings/ESM2/final_embeddings_colmean/phage_rbp_esm2_colmean_embeddings.csv
Host similarity : datasets/121strains_74phages/cosine_similarity/ESM2/host_cosine_similarity_ESM2_centered_0to1_121strains_74phages_canonical.csv
Phage similarity: datasets/121strains_74phages/cosine_similarity/ESM2/phage_cosine_similarity_ESM2_centered_0to1_121strains_74phages_canonical.csv

ESMC
Host embeddings : datasets/121strains_74phages/embeddings/ESMC/final_embeddings_colmean/host_klocus_esmc_colmean_embeddings.csv
Phage embeddings: datasets/121strains_74phages/embeddings/ESMC/final_embeddings_colmean/phage_rbp_esmc_colmean_emb